In [ ]:
import numpy as np
import pandas as pd
import pickle
from sklearn.metrics import classification_report, confusion_matrix,fbeta_score,precision_recall_curve,auc
from sklearn.preprocessing import StandardScaler


In [9]:
base_model = "../models/xgboost_baseline.pkl"
model_2 = "../models/xgboost_f2.pkl"
with open(base_model, "rb") as f:
    model = pickle.load(f)
    
with open(model_2, "rb") as f:
    model_2 = pickle.load(f)

print(type(model))
print(model)

<class 'xgboost.sklearn.XGBClassifier'>
XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=50,
              enable_categorical=False, eval_metric='aucpr', feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=1000, n_jobs=-1,
              num_parallel_tree=None, random_state=42, ...)


In [11]:
print("Number of features:", model.n_features_in_)

print("\nFeature names:")
print(model.feature_names_in_)
print(model_2.feature_names_in_)

Number of features: 19

Feature names:
['airspeed_cmd' 'airspeed_meas' 'airspeed_error' 'roll_cmd' 'roll_meas'
 'roll_error' 'pitch_cmd' 'pitch_meas' 'pitch_error' 'yaw_cmd' 'yaw_meas'
 'yaw_error_clean' 'vel_x' 'vel_y' 'vel_z' 'imu_x' 'imu_y' 'imu_z'
 'ground_speed']
['airspeed_cmd' 'airspeed_meas' 'airspeed_error' 'roll_cmd' 'roll_meas'
 'roll_error' 'pitch_cmd' 'pitch_meas' 'pitch_error' 'yaw_cmd' 'yaw_meas'
 'yaw_error_clean' 'vel_x' 'vel_y' 'vel_z' 'imu_x' 'imu_y' 'imu_z'
 'ground_speed']


In [13]:
def wrap_angle_deg(angle):
    """
    Normalize angular error to [-180, 180] degrees.
    """
    return (angle + 180) % 360 - 180


def preprocess(df):
    """
    Prepare raw UAV data for the saved XGBoost baseline model.

    Returns
    -------
    clean_df : DataFrame
        Cleaned dataframe with engineered features.

    X : DataFrame
        Exactly the 19 features expected by the saved model.

    y : Series
        Binary fault label.
    """

    clean_df = df.copy()

    # ==========================================
    # 1. HANDLE MISSING VALUES
    # ==========================================

    numeric_cols = clean_df.select_dtypes(
        include=[np.number]
    ).columns

    # Time-series forward fill
    clean_df[numeric_cols] = clean_df[numeric_cols].ffill()

    # Fill values at the beginning of a flight/dataset
    clean_df[numeric_cols] = clean_df[numeric_cols].bfill()

    # ==========================================
    # 2. AIRSPEED ERROR
    # ==========================================

    clean_df["airspeed_error"] = (
        clean_df["airspeed_cmd"]
        - clean_df["airspeed_meas"]
    )

    # ==========================================
    # 3. ROLL ERROR
    # ==========================================

    clean_df["roll_error"] = (
        clean_df["roll_cmd"]
        - clean_df["roll_meas"]
    )

    # ==========================================
    # 4. PITCH ERROR
    # ==========================================

    clean_df["pitch_error"] = (
        clean_df["pitch_cmd"]
        - clean_df["pitch_meas"]
    )

    # ==========================================
    # 5. YAW ERROR
    # ==========================================

    clean_df["yaw_error_clean"] = wrap_angle_deg(
        clean_df["yaw_cmd"]
        - clean_df["yaw_meas"]
    )

    # ==========================================
    # 6. GROUND SPEED
    # ==========================================

    clean_df["ground_speed"] = np.sqrt(
        clean_df["vel_x"] ** 2 +
        clean_df["vel_y"] ** 2 +
        clean_df["vel_z"] ** 2
    )

    # ==========================================
    # 7. BINARY FAULT
    # ==========================================

    clean_df["binary_fault"] = (
        (clean_df["engine_fault"] > 0) |
        (clean_df["aileron_fault"] > 0) |
        (clean_df["elevator_fault"] > 0) |
        (clean_df["rudder_fault"] > 0)
    ).astype(int)

    # ==========================================
    # 8. EXACT MODEL FEATURES
    # ==========================================

    feature_cols = [
        "airspeed_cmd",
        "airspeed_meas",
        "airspeed_error",

        "roll_cmd",
        "roll_meas",
        "roll_error",

        "pitch_cmd",
        "pitch_meas",
        "pitch_error",

        "yaw_cmd",
        "yaw_meas",
        "yaw_error_clean",

        "vel_x",
        "vel_y",
        "vel_z",

        "imu_x",
        "imu_y",
        "imu_z",

        "ground_speed"
    ]

    X = clean_df[feature_cols].copy()

    y = clean_df["binary_fault"].copy()

    return clean_df, X, y

In [14]:
raw_data = pd.read_csv("../data/test_raw.csv")

In [22]:
# 1. Preprocess raw test data
clean_df_test, X_test, y_fault_test = preprocess(raw_data)
flight_id_test = clean_df_test['bag']

# 2. Transform test data using the existing scaler (DO NOT call fit_transform)
scaler = StandardScaler()
df_test_scaled = scaler.transform(X_test)

NotFittedError: This StandardScaler instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.

In [ ]:
y_pred = model.predict(X_scale)
y_prob = model.predict_proba(X_scale)[:, 1]
print("=== Classification Report ===")
print(classification_report(y, y_pred))

print("\n=== Confusion Matrix ===")
print(confusion_matrix(y, y_pred))


=== Classification Report ===
              precision    recall  f1-score   support

           0       0.92      0.34      0.50     41135
           1       0.17      0.81      0.28      6740

    accuracy                           0.41     47875
   macro avg       0.54      0.58      0.39     47875
weighted avg       0.81      0.41      0.47     47875


=== Confusion Matrix ===
[[14126 27009]
 [ 1250  5490]]


In [ ]:
best_thresh = 0.56
# --- 1. Preprocess & Scale Test Data ---
clean_df_test, X_test, y_fault_test = preprocess(raw_test_data)
flight_id_test = clean_df_test['bag']

df_test_scaled = scaler.transform(X_test)

# --- 2. Predict Probabilities ---
test_probs_raw = model_2.predict_proba(df_test_scaled)[:, 1]

# --- 3. 10-Frame Rolling Window Mean ---
test_df = pd.DataFrame({
    'flight_id': flight_id_test.values,
    'raw_probs': test_probs_raw
})

test_probs_smooth = test_df.groupby('flight_id')['raw_probs'].transform(
    lambda x: x.rolling(window=10, min_periods=1).mean()
).values

# --- 4. Apply Validation Threshold & Evaluate ---
test_preds = (test_probs_smooth >= best_thresh).astype(int)

p_test, r_test, _ = precision_recall_curve(y_fault_test, test_probs_smooth)
test_pr_auc = auc(r_test, p_test)
test_f2 = fbeta_score(y_fault_test, test_preds, beta=2.0, zero_division=0)

print(f"---> Applied Threshold: {best_thresh:.2f}")
print(f"---> Smoothed Test PR-AUC: {test_pr_auc:.4f}")
print(f"---> Test F2-Score: {test_f2:.4f}\n")

print("=== Test Classification Report ===")
print(classification_report(y_fault_test, test_preds, target_names=['Healthy (0)', 'Faulty (1)']))

print("=== Test Confusion Matrix ===")
print(confusion_matrix(y_fault_test, test_preds))

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.92      0.34      0.50     41135
           1       0.17      0.81      0.28      6740

    accuracy                           0.41     47875
   macro avg       0.54      0.58      0.39     47875
weighted avg       0.81      0.41      0.47     47875


=== Confusion Matrix ===
[[14126 27009]
 [ 1250  5490]]

F2 Score: 0.4617
